# 01. Planner 테스트

이 노트북은 Planner가 사용자 질문을 어떻게 분석하고 실행 계획을 생성하는지 테스트한다.

## 확인할 사항
- 다양한 질문 유형에 대해 올바른 query_type이 선택되는지
- steps가 논리적으로 구성되는지
- JSON 파싱과 검증이 올바르게 동작하는지

## 실행 전 준비사항
- `config/config.yaml` 설정 확인 (llm_provider, ollama/openai 설정)
- Ollama 또는 OpenAI API 접근 가능 확인

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"프로젝트 루트: {project_root}")

## 설정 및 PlannerService 초기화

In [ ]:
from src.db_to_llm.shared.config.config_loader import load_config
from src.db_to_llm.shared.llm.llm_factory import create_llm_client
from src.db_to_llm.shared.logging.logger import setup_logger
from src.db_to_llm.stream.planner.planner_service import PlannerService
from src.db_to_llm.stream.prompts.prompt_manager import PromptManager

config_path = project_root / "config" / "config.yaml"
config = load_config(config_path)
setup_logger(log_level="INFO")

# LLM 클라이언트와 PromptManager 생성
llm_client = create_llm_client(config)
prompt_manager = PromptManager()  # 기본 경로 사용
planner = PlannerService(llm_client=llm_client, prompt_manager=prompt_manager)

print(f"LLM Provider: {llm_client.provider_name}")
print(f"사용 가능한 프롬프트 키: {prompt_manager.list_prompt_keys()}")

## 테스트 1: DB_ONLY 질문

In [ ]:
import json

question_db = "지난달 매출 상위 10개 제품을 알려줘"
print(f"질문: {question_db}")
print("-" * 50)

plan = planner.plan_question(question_db)
print(json.dumps(plan.to_dict(), ensure_ascii=False, indent=2))
print(f"\n결과: query_type={plan.query_type}, is_composite={plan.is_composite}")

## 테스트 2: RAG_ONLY 질문

In [ ]:
question_rag = "오류 코드 E1003의 원인과 해결방법이 뭐야?"
print(f"질문: {question_rag}")
print("-" * 50)

plan = planner.plan_question(question_rag)
print(json.dumps(plan.to_dict(), ensure_ascii=False, indent=2))

## 테스트 3: DB_THEN_RAG 복합 질문

In [ ]:
question_complex = "지난달 가장 많이 발생한 오류 코드를 조회하고, 그 오류의 원인과 해결방법도 알려줘"
print(f"질문: {question_complex}")
print("-" * 50)

plan = planner.plan_question(question_complex)
print(json.dumps(plan.to_dict(), ensure_ascii=False, indent=2))

## 테스트 4: GENERAL 질문

In [ ]:
question_general = "SQL의 INNER JOIN과 LEFT JOIN의 차이점이 뭐야?"
print(f"질문: {question_general}")
print("-" * 50)

plan = planner.plan_question(question_general)
print(json.dumps(plan.to_dict(), ensure_ascii=False, indent=2))

## Planner 검증기 직접 테스트

In [ ]:
from src.db_to_llm.stream.planner.plan_validator import PlanValidationError, validate_plan_payload

# 올바른 plan
valid_plan = {
    "is_composite": False,
    "query_type": "DB_ONLY",
    "steps": [{"step": 1, "type": "db", "goal": "매출 데이터 조회", "depends_on": []}]
}
validate_plan_payload(valid_plan)
print("유효한 plan: 통과")

# 잘못된 query_type
try:
    invalid_plan = {**valid_plan, "query_type": "INVALID_TYPE"}
    validate_plan_payload(invalid_plan)
except PlanValidationError as e:
    print(f"잘못된 query_type 감지: {e}")